# New Observation

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import chi2
from astroquery.gaia import Gaia
import numpy as np
from astropy.time import Time

from thejoker import JokerPrior, TheJoker, RVData
from thejoker.plot import plot_rv_curves
import astropy.units as u
import pymc as pm
import exoplanet.units as xu

Status messages could not be retrieved


/Users/mac/Library/Python/3.10/lib/python/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


In [2]:
candidates_RVs = pd.read_csv("Data/RV_Gaia_LAMOST.csv")

In [3]:
Desig = ['J065401.91+752725.9', 'J065401.91+752725.9', 'J065401.91+752725.9', 'J065401.91+752725.9', 'J065401.91+752725.9', 'J000556.86-012835.5', 'J000556.86-012835.5', 'J000556.86-012835.5', 'J000556.86-012835.5', 'J000556.86-012835.5']
t = [60562.75,60704.66,60716.90,60728.85,60735.96, 60605.75,60664.58,60943.64,60992.53, 61172.93]
rv = [-337.5,-324.4,-344.45,-325.58, -331.7, -110.6,-113.8,-107.1,-109.8, -118.3]
err = [1.1,0.8,0.6,0.7,0.8,0.6,0.3,0.3,0.2, 0.4]
labels = ['APF', 'APF', 'APF', 'APF', 'APF', 'APF', 'MIKE', 'MIKE', 'MIKE', 'MIKE']

df_obs_new = pd.DataFrame({
	"designation":Desig, 
	"radial_velocity": rv,
	"radial_velocity_error":err,
	"epoch": t, 
	"label": labels
})
df_RVs_all = pd.concat([candidates_RVs, df_obs_new], ignore_index = True).sort_values(by = 'designation')
df_RVs_all.to_csv("RV_all.csv")

# thejoker Samples

## J000556 Sampling

In [4]:
#INCLUDING MIKE OBSERVATIONS, APF, AND LAMOST MRS
t = [58452,59899,60605.75,60664.58,60943.64,60992.53, 61172.93]
rv = [-117.47,-116.64,-110.6,-113.8,-107.1,-109.8, -118.3] * u.km/u.s
err = [1.01,0.90,0.6,0.3,0.3,0.2,0.4] * u.km/u.s

data = RVData(t=t, rv=rv, rv_err=err)

#ORIGINAL JOKER PRIOR
#prior = JokerPrior.default(P_min=50*u.day, P_max=256*u.day,
#                           sigma_K0=50*u.km/u.s, sigma_v=25*u.km/u.s)

#CHANGE SYSTEMIC VELOCITY PRIOR TO A GAUSSIAN CENTERED AT 20 KM/S WITH SIGMA=25 KM/S
with pm.Model() as model:
#    v0 = xu.with_unit(pm.Normal('v0', 20., 25.),
#                     u.km/u.s)

	prior = JokerPrior.default(P_min=3*u.day, P_max=2000*u.day,
		sigma_K0=50*u.km/u.s,sigma_v=125*u.km/u.s)
#        pars={'v0': v0})

	
joker = TheJoker(prior)

prior_samples = prior.sample(size=1_000_000)
samples_J000556 = joker.rejection_sample(data, prior_samples)

In [ ]:
samples_J000556 = samples_J000556[0]
samples_J000556["P"][0].value

## J065401 Sampling

In [ ]:
#INCLUDING MIKE OBSERVATIONS, APF, AND LAMOST MRS

t = [57382.84,58437.00,60562.75,60704.66,60716.90,60728.85, 60735.96]
rv = [-335.4,-317.1,-337.5,-324.4,-344.45,-325.58, -331.7] * u.km/u.s
err = [2.3,1.2,1.1,0.8,0.6,0.7,0.8] * u.km/u.s

data = RVData(t=t, rv=rv, rv_err=err)

#ORIGINAL JOKER PRIOR
#prior = JokerPrior.default(P_min=50*u.day, P_max=256*u.day,
#                           sigma_K0=50*u.km/u.s, sigma_v=25*u.km/u.s)

#CHANGE SYSTEMIC VELOCITY PRIOR TO A GAUSSIAN CENTERED AT 20 KM/S WITH SIGMA=25 KM/S
with pm.Model() as model:
#    v0 = xu.with_unit(pm.Normal('v0', 20., 25.),
#                     u.km/u.s)

	prior = JokerPrior.default(P_min=3*u.day, P_max=100*u.day,
		sigma_K0=50*u.km/u.s,sigma_v=125*u.km/u.s)
#        pars={'v0': v0})


joker = TheJoker(prior)

prior_samples = prior.sample(size=10_000_000)
samples_J065401 = joker.rejection_sample(data, prior_samples)

In [ ]:
samples_J065401['P'].value